# RankNet

In this notebook, a neural network is built using PyTorch.

The neural network learns how to rank candidates based on their similarity to a target job title. The initial rankings are those determined by an SBERT model. Then the neural network utilises RankNet architecture to be able to re-rank candidates based on a manual human input.

A human is able to manually 'star' any candidates they wish to be considered as ideal matches, and the neural network will then learn to favour these candidates over others.

In [156]:
# import required libraries
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim
from itertools import combinations
from sklearn.model_selection import train_test_split

In [157]:
# remove limit of column width display to allow full job titles to be viewed
pd.set_option('display.max_colwidth', None)

In [158]:
# import the data from the pickle file that was saved in the previous notebook
data = pd.read_pickle("../data/potential_talents_ranked.pkl")

In [159]:
# view data
data.sort_values(by='sbert_rank_q1', ascending=True)

,id,job_title,sbert_similarity_q1,sbert_rank_q1,sbert_similarity_q2,sbert_rank_q2,embeddings
5,3,aspiring hr professional,0.803918,1,0.633336,6,"[-0.032186154, 0.07609709, -0.026287533, 0.13310567, 0.028368013, -0.035839733, 0.023116007, 0.00510067, -0.048712455, 0.06301331, -0.06577844, 0.0005947313, -0.013079674, -0.007093651, 0.00871463, 0.042407386, -0.02234534, -0.008893932, 0.07943562, -0.031224119, -0.055657446, -0.005149189, 0.005027218, -0.117027596, -0.018567972, -0.0029963637, -0.003447877, 0.069010206, -0.061891947, -0.0348628, -0.026487213, -0.031997584, 0.009918475, 0.02948102, 0.042366087, 0.063924715, 0.06607231, 0.00718141, 0.023348693, 0.06359001, -0.05071547, -0.06447675, -0.059018668, -0.025497934, -0.033814535, -0.07019521, 0.021864787, -0.0027895211, -0.01622276, -0.008471565, -0.033151295, -0.077612616, 0.07251939, 0.021094754, 0.02112421, -0.02691429, 0.017322622, -0.012442623, -0.042570677, -0.039015636, -0.083173834, -0.04966549, -0.048233375, -0.03799064, 0.050456002, -0.079469144, -0.059242897, 0.045883287, 0.024357175, -0.053546265, -0.047789928, -0.06713471, -0.08829304, 0.03139624, 0.108263634, -0.06365079, 0.011315225, 0.003104665, 0.056971908, -0.018120058, 0.059851628, 0.008164984, -0.060525823, 0.1140431, -0.058058172, -0.02483074, -0.006533914, 0.04277528, 0.014141435, 0.0060619605, 0.041829005, 0.011834393, -0.016596673, 0.0017009495, -0.0047547533, 0.016165935, -0.02306667, 0.09089491, -0.032160994, 0.07479696, ...]"
6,97,aspiring hr professional,0.803918,1,0.633336,6,"[-0.032186154, 0.07609709, -0.026287533, 0.13310567, 0.028368013, -0.035839733, 0.023116007, 0.00510067, -0.048712455, 0.06301331, -0.06577844, 0.0005947313, -0.013079674, -0.007093651, 0.00871463, 0.042407386, -0.02234534, -0.008893932, 0.07943562, -0.031224119, -0.055657446, -0.005149189, 0.005027218, -0.117027596, -0.018567972, -0.0029963637, -0.003447877, 0.069010206, -0.061891947, -0.0348628, -0.026487213, -0.031997584, 0.009918475, 0.02948102, 0.042366087, 0.063924715, 0.06607231, 0.00718141, 0.023348693, 0.06359001, -0.05071547, -0.06447675, -0.059018668, -0.025497934, -0.033814535, -0.07019521, 0.021864787, -0.0027895211, -0.01622276, -0.008471565, -0.033151295, -0.077612616, 0.07251939, 0.021094754, 0.02112421, -0.02691429, 0.017322622, -0.012442623, -0.042570677, -0.039015636, -0.083173834, -0.04966549, -0.048233375, -0.03799064, 0.050456002, -0.079469144, -0.059242897, 0.045883287, 0.024357175, -0.053546265, -0.047789928, -0.06713471, -0.08829304, 0.03139624, 0.108263634, -0.06365079, 0.011315225, 0.003104665, 0.056971908, -0.018120058, 0.059851628, 0.008164984, -0.060525823, 0.1140431, -0.058058172, -0.02483074, -0.006533914, 0.04277528, 0.014141435, 0.0060619605, 0.041829005, 0.011834393, -0.016596673, 0.0017009495, -0.0047547533, 0.016165935, -0.02306667, 0.09089491, -0.032160994, 0.07479696, ...]"
7,6,aspiring hr specialist,0.719585,3,0.596391,8,"[-0.037710946, 0.07939161, -0.02258595, 0.12964943, 0.008232193, -0.038806528, 0.020269185, 0.023260167, -0.08061382, 0.014234447, -0.0723415, 0.012547339, -0.02544814, 0.011474871, -0.004061834, 0.017563703, 0.0032807402, 0.0021597396, 0.06313477, -0.06082017, -0.052388307, 0.0048186434, 0.016573753, -0.10442197, -0.013030899, -0.03149833, 0.00459803, 0.049903728, -0.060396623, -0.011197143, 0.005811984, -0.05341136, -0.040253133, 0.03515007, 0.02113885, 0.07585503, 0.0689005, -0.011668077, -0.025629407, 0.050472677, -0.056051735, -0.06266889, -0.07564953, -0.00895485, -0.0100163175, -0.067963086, 0.022741852, -0.015288676, -0.004305438, -0.01563949, -0.06937195, -0.102506414, 0.059613027, 0.01138245, 0.0066652503, -0.03916251, 0.027047569, -0.030351182, -0.08014833, -0.021018468, -0.061419576, -0.054714583, -0.030995466, -0.051873967, 0.019727375, -0.051398415, -0.039012354, 0.0353725, 0.048276447, -0.032032344, -0.027875034, -0.05563027, -0.075957514, 0.06170238, 0.09661184, -0.03438414, 0.022460649, 0.021637116, 0.08504388, -0.027164396, 0.05694489, 0.009290631, -

In [160]:
# create the neural network
class RankNet(nn.Module):

    def __init__(self, input_size):
        super().__init__()

        # add the layers: linear layers with ReLU activation function and 20% dropout between each one
        self.network = nn.Sequential(

            nn.Linear(input_size, 400),
            nn.ReLU(),
            nn.Dropout(p=0.2),

            nn.Linear(400, 800),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            
            nn.Linear(800, 200),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            
            nn.Linear(200, 1)
        )

    # create the forward step of the neural network, taking in 2 candidates and returning the difference in their scores
    def forward(self, x1, x2):
        score_1 = self.network(x1)
        score_2 = self.network(x2)

        score_difference = score_1 - score_2
        return score_difference

In [161]:
# display the initial ranking created by SBERT in relation to query 1
print('Candidates ranked by their similarity to query 1: Aspiring Human Resources')
data[['id', 'job_title', 'sbert_rank_q1']].sort_values(by='sbert_rank_q1', ascending=True)

Candidates ranked by their similarity to query 1: Aspiring Human Resources


,id,job_title,sbert_rank_q1
5,3,aspiring hr professional,1
6,97,aspiring hr professional,1
7,6,aspiring hr specialist,3
0,28,seeking hr opportunities,4
9,7,student at humber college and aspiring hr generalist,5
2,99,seeking hr position,6
4,67,hr staffing and recruiting professional,7
8,74,hr professional,8
3,10,seeking hr hris and generalist positions,9
15,82,aspiring hr professional an energetic and team focused leader,10


In [178]:
# create an input so that candidates can be manually reranked by starring those that should be at the top

# empty list
q1_starred_ids = []

# create new column of similarity scores to be fed into the neural network
data['modelling_similarity_q1'] = data['sbert_similarity_q1']
# ask if there are any candidates to star
star_candidates_q1 = input("Would you like to star any candidates for query 1? (y/n)")
# if there are
if star_candidates_q1.lower() == 'y':
    # enter the candidate ids with comma separator
    q1_starred_ids = [int(idx.strip()) for idx in input("Enter the candidate IDs separated by commas:").split(",")]
    # locate the candidates with those ids and update their similarity score to be 1 (maximum)
    for idx in q1_starred_ids:
        data.loc[data['id'] == idx, 'modelling_similarity_q1'] = 1
    print("IDs starred for query 1:", q1_starred_ids)
# if there are no candidates to be starred, do nothing
else:
    print('No changes required for query 1.')

Would you like to star any candidates for query 1? (y/n) y
Enter the candidate IDs separated by commas: 10, 45


IDs starred for query 1: [10, 45]


In [163]:
# display the initial ranking created by SBERT in relation to query 2
data[['id', 'job_title', 'sbert_rank_q2']].sort_values(by='sbert_rank_q2', ascending=True)

,id,job_title,sbert_rank_q2
0,28,seeking hr opportunities,1
1,94,seeking human resources opportunities open to travel and relocation,2
2,99,seeking hr position,3
3,10,seeking hr hris and generalist positions,4
4,67,hr staffing and recruiting professional,5
5,3,aspiring hr professional,6
6,97,aspiring hr professional,6
7,6,aspiring hr specialist,8
8,74,hr professional,9
9,7,student at humber college and aspiring hr generalist,10


In [179]:
# create an input so that candidates can be manually reranked by starring those that should be at the top

# empty list
q2_starred_ids = []

# create new column of similarity scores to be fed into the neural network
data['modelling_similarity_q2'] = data['sbert_similarity_q2']
# ask if there are any candidates to star
star_candidates_q2 = input("Would you like to star any candidates for query 2? (y/n)")
# if there are
if star_candidates_q2.lower() == 'y':
    # enter the candidate ids with comma separator
    q2_starred_ids = [int(idx.strip()) for idx in input("Enter the candidate IDs separated by commas:").split(",")]
    # locate the candidates with those ids and update their similarity score to be 1 (maximum)
    for idx in q2_starred_ids:
        data.loc[data['id'] == idx, 'modelling_similarity_q2'] = 1
    print("IDs starred for query 2:", q2_starred_ids)
# if there are no candidates to be starred, do nothing
else:
    print('No changes required for query 2.')

Would you like to star any candidates for query 2? (y/n) y
Enter the candidate IDs separated by commas: 20, 90


IDs starred for query 2: [20, 90]


In [180]:
# display the data with the original similarity scores alongside the updated ones, showing starred candidates with their new score of 1
data[['id', 'job_title', 'sbert_similarity_q1', 'modelling_similarity_q1', 'sbert_similarity_q2', 'modelling_similarity_q2']].sort_values(by='modelling_similarity_q2', ascending=False)

,id,job_title,sbert_similarity_q1,modelling_similarity_q1,sbert_similarity_q2,modelling_similarity_q2
36,90,undergraduate research assistant at styczynski lab,0.369043,0.369043,0.312178,1.000000
0,28,seeking hr opportunities,0.699635,0.699635,0.746315,0.746315
1,94,seeking human resources opportunities open to travel and relocation,0.564194,0.564194,0.709997,0.709997
2,99,seeking hr position,0.673482,0.673482,0.693969,0.693969
3,10,seeking hr hris and generalist positions,0.638313,1.000000,0.693324,0.693324
4,67,hr staffing and recruiting professional,0.663799,0.663799,0.636295,0.636295
5,3,aspiring hr professional,0.803918,0.803918,0.633336,0.633336
6,97,aspiring hr professional,0.803918,0.803918,0.633336,0.633336
7,6,aspiring hr specialist,0.719585,0.719585,0.596391,0.596391
8,74,hr professional,0.646536,0.646536,0.582953,0.582953


In [181]:
# creating pairs of candidates for query 1

# defining the number of candidatea
num_candidates = len(data)
# defining how many sample pairs to create
num_sample_pairs = 4000

# generate 2 lists of random numbers within the range of the number of candidates
random_i = np.random.randint(0, num_candidates, num_sample_pairs)
random_j = np.random.randint(0, num_candidates-1, num_sample_pairs)
# ensure there are no pairs containing the same candidate twice by ensuring j != i
random_j = random_j + (random_j >= random_i)

# combine these 2 lists into a list of pairs
random_pairs = list(zip(random_i, random_j))

# extract the embeddings and similarities from the data (after starring)
embeddings = data['embeddings'].values
model_sim_q1 = data['modelling_similarity_q1'].values

In [182]:
# create the pairwise scores for candidates

# create empty lists for the embeddings of each candidate in the pair and for their comparative score
x_i = []
x_j = []
output_ij = []

# for each pair
for i, j in random_pairs:

    # take the embeddings of the first candidate in the pair, then the second
    x_i.append(embeddings[i])
    x_j.append(embeddings[j])

    # if the first has higher similarity, score = 1
    if model_sim_q1[i] > model_sim_q1[j]:
        result_ij = 1.0
    # if their similarities match, score = 0.5
    elif model_sim_q1[i] == model_sim_q1[j]:
        result_ij = 0.5
    # if the second has higher similarity, score = 0
    elif model_sim_q1[i] < model_sim_q1[j]:
        result_ij = 0.0

    # record the score of this pair
    output_ij.append(result_ij)

In [183]:
# turn the lists into tensors so they can be fed into the neural network
x_i = torch.tensor(x_i, dtype=torch.float32)
x_j = torch.tensor(x_j)
output_ij = torch.tensor(output_ij)

In [184]:
# split the many pairs (with their scores) into train and validation data
xi_train, xi_val, xj_train, xj_val, outij_train, outij_val = train_test_split(
    x_i, x_j, output_ij, test_size=0.2)

outij_train = outij_train.unsqueeze(1)
outij_val = outij_val.unsqueeze(1)

In [185]:
# define the input size, used as the number of nodes in the first layer of the neural network
input_size = len(data['embeddings'][0])
# instantiate the model defined earlier
model = RankNet(input_size=input_size)
# define which loss function to use
criterion = nn.BCEWithLogitsLoss()
# define which optimiser to use and the learning rate
optimiser = optim.Adam(model.parameters(), lr=0.0005)

In [186]:
# define how many times to pass through the neural network
epochs = 1000
# start the loss at infinity
best_loss = float('inf')
# define how many epochs to continue with no improvement in loss before stopping
patience = 10

# train the model
for epoch in range(epochs):
    # put the model in train mode
    model.train()

    # reset the gradients of the optimiser to zero
    optimiser.zero_grad()

    # generate the model score difference between a pair of candidates
    score_difference = model(xi_train, xj_train)

    # calculate the loss based on whether the score difference is comensurate with the given score (label) above
    loss = criterion(score_difference, outij_train)

    # update the model parameters
    loss.backward()
    optimiser.step()

    # put the model in evaluation mode
    model.eval()

    # tell the model that we no longer want to train it
    with torch.no_grad():
        # assess how the model does on unseen data
        val_difference = model(xi_val, xj_val)
    
        val_loss = criterion(
            val_difference,
            outij_val
        )

    # tell the model to stop if the loss doesn't improve for the defined number of epochs above (patience)
    if val_loss.item() < best_loss:
        best_loss = val_loss.item()
        counter = 0

        best_model = {
            key: value.detach().clone()
            for key, value in model.state_dict().items()
        }
    else:
        counter += 1

    # every 10 epochs, print loss
    if (epoch+1) % 10 == 0:
        print(
            f"Epoch [{epoch+1} / {epochs}], "
            f"Training Loss: {loss.item():.4f}, "
            f"Validation Loss: {val_loss.item():.4f}"
        )

    if counter >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

# revert the model back to its best state
model.load_state_dict(best_model)

Epoch [10 / 1000], Training Loss: 0.6060, Validation Loss: 0.5830
Epoch [20 / 1000], Training Loss: 0.2888, Validation Loss: 0.2719
Epoch [30 / 1000], Training Loss: 0.1202, Validation Loss: 0.1063
Epoch [40 / 1000], Training Loss: 0.0705, Validation Loss: 0.0507
Epoch [50 / 1000], Training Loss: 0.0657, Validation Loss: 0.0358
Epoch [60 / 1000], Training Loss: 0.0505, Validation Loss: 0.0316
Epoch [70 / 1000], Training Loss: 0.0465, Validation Loss: 0.0270
Epoch [80 / 1000], Training Loss: 0.0447, Validation Loss: 0.0234
Epoch [90 / 1000], Training Loss: 0.0385, Validation Loss: 0.0216
Epoch [100 / 1000], Training Loss: 0.0350, Validation Loss: 0.0224
Early stopping at epoch 103


<All keys matched successfully>

In [189]:
# convert the embeddings into a tensor
input_array = torch.tensor(data['embeddings'])

# put the model in evaluation mode
model.eval()

# generate the scores of the candidates
with torch.no_grad():
    scores = model.network(input_array)

# rank the candidates based off the neural network scores
data['ranknet_scores_q1'] = scores.numpy().flatten()
data['ranknet_rank_q1'] = data['ranknet_scores_q1'].rank(ascending=False, method='min').astype(int)

In [190]:
# display the rankings
data[['id', 'job_title', 'ranknet_rank_q1']].sort_values(by=('ranknet_rank_q1'), ascending=True)

,id,job_title,ranknet_rank_q1
5,3,aspiring hr professional,1
6,97,aspiring hr professional,1
3,10,seeking hr hris and generalist positions,3
7,6,aspiring hr specialist,4
0,28,seeking hr opportunities,5
2,99,seeking hr position,6
9,7,student at humber college and aspiring hr generalist,7
4,67,hr staffing and recruiting professional,8
8,74,hr professional,9
15,82,aspiring hr professional an energetic and team focused leader,10


In [191]:
# creating pairs of candidates for query 2

# defining the number of candidatea
num_candidates = len(data)
# defining how many sample pairs to create
num_sample_pairs = 4000

# generate 2 lists of random numbers within the range of the number of candidates
random_i = np.random.randint(0, num_candidates, num_sample_pairs)
random_j = np.random.randint(0, num_candidates-1, num_sample_pairs)
# ensure there are no pairs containing the same candidate twice by ensuring j != i
random_j = random_j + (random_j >= random_i)

# combine these 2 lists into a list of pairs
random_pairs = list(zip(random_i, random_j))

# extract the embeddings and similarities from the data (after starring)
embeddings = data['embeddings'].values
model_sim_q2 = data['modelling_similarity_q2'].values

x_i = []
x_j = []
output_ij = []

for i, j in random_pairs:

    x_i.append(embeddings[i])
    x_j.append(embeddings[j])

    if model_sim_q2[i] > model_sim_q2[j]:
        result_ij = 1.0
    elif model_sim_q2[i] == model_sim_q2[j]:
        result_ij = 0.5
    elif model_sim_q2[i] < model_sim_q2[j]:
        result_ij = 0.0

    output_ij.append(result_ij)

x_i = torch.tensor(x_i)
x_j = torch.tensor(x_j)
output_ij = torch.tensor(output_ij)

xi_train, xi_val, xj_train, xj_val, outij_train, outij_val = train_test_split(
    x_i, x_j, output_ij, test_size=0.2)

outij_train = outij_train.unsqueeze(1)
outij_val = outij_val.unsqueeze(1)

In [192]:
input_size = len(data['embeddings'][0])
model_2 = RankNet(input_size=input_size)
criterion = nn.BCEWithLogitsLoss()
optimiser = optim.Adam(model_2.parameters(), lr=0.001)

epochs = 1000
best_loss = float('inf')
patience = 10

for epoch in range(epochs):
    model_2.train()

    optimiser.zero_grad()

    score_difference = model_2(xi_train, xj_train)

    loss = criterion(score_difference, outij_train)

    loss.backward()

    optimiser.step()

    model_2.eval()

    with torch.no_grad():
    
        val_difference = model_2(xi_val, xj_val)
    
        val_loss = criterion(
            val_difference,
            outij_val
        )

    if val_loss.item() < best_loss:
        best_loss = val_loss.item()
        counter = 0

        best_model = {
            key: value.detach().clone()
            for key, value in model_2.state_dict().items()
        }
    else:
        counter += 1

    if (epoch+1) % 10 == 0:
        print(
            f"Epoch [{epoch+1} / {epochs}], "
            f"Training Loss: {loss.item():.4f}, "
            f"Validation Loss: {val_loss.item():.4f}"
        )

    if counter >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

model_2.load_state_dict(best_model)

Epoch [10 / 1000], Training Loss: 0.3757, Validation Loss: 0.3380
Epoch [20 / 1000], Training Loss: 0.1111, Validation Loss: 0.0937
Epoch [30 / 1000], Training Loss: 0.0646, Validation Loss: 0.0358
Epoch [40 / 1000], Training Loss: 0.0534, Validation Loss: 0.0257
Epoch [50 / 1000], Training Loss: 0.0467, Validation Loss: 0.0261
Early stopping at epoch 54


<All keys matched successfully>

In [193]:
input_array = torch.tensor(data['embeddings'])

model_2.eval()

with torch.no_grad():
    scores = model_2.network(input_array)

data['ranknet_scores_q2'] = scores.numpy().flatten()
data['ranknet_rank_q2'] = data['ranknet_scores_q2'].rank(ascending=False, method='min').astype(int)

In [194]:
data[['id', 'job_title', 'ranknet_rank_q2']].sort_values(by=('ranknet_rank_q2'), ascending=True)

,id,job_title,ranknet_rank_q2
36,90,undergraduate research assistant at styczynski lab,1
1,94,seeking human resources opportunities open to travel and relocation,2
0,28,seeking hr opportunities,3
2,99,seeking hr position,4
3,10,seeking hr hris and generalist positions,5
4,67,hr staffing and recruiting professional,6
5,3,aspiring hr professional,7
6,97,aspiring hr professional,7
7,6,aspiring hr specialist,9
8,74,hr professional,10
